In [ ]:
from pathlib import Path

import torch


model_dir = Path.cwd()
if not (model_dir / "model_best.pth").is_file():
    model_dir = model_dir / "exp/s3dis/ptv3-area5/model"
source = model_dir / "model_best.pth"
output = model_dir / "model_best_fp16.pth"

checkpoint = torch.load(source, map_location="cpu", weights_only=False)
state = checkpoint["state_dict"]

fp16_state = state.copy()
for name, tensor in state.items():
    if tensor.is_floating_point():
        fp16_state[name] = tensor.half()


torch.save({"epoch": checkpoint["epoch"], "state_dict": fp16_state}, output)

saved = torch.load(output, map_location="cpu", weights_only=True)["state_dict"]
assert saved.keys() == state.keys()
for name, original in state.items():
    expected = original.half() if original.is_floating_point() else original
    assert saved[name].dtype == expected.dtype and torch.equal(saved[name], expected), name

print(f"FP16 checkpoint: {output}")
print(f"File size: {output.stat().st_size / 2**20:.2f} MiB")
print(f"FP16 tensors: {sum(t.dtype == torch.float16 for t in saved.values())}/{len(saved)}")
